# UK CPIH Inflation Analyser
### Q&A over ONS CPIH data (series-200526.csv)
_Source: Office for National Statistics — Consumer Prices Index including owner occupiers' housing costs (CPIH)_

**Run all cells** (`Runtime → Run all`) to launch the Gradio UI at the bottom.

In [41]:

!pip install -q gradio groq matplotlib pandas

In [43]:
import os, re, io, base64, textwrap
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from groq import Groq
import gradio as gr
import getpass


# Get a free API key at https://console.groq.com
# When prompted below after execution in runtime , paste your key — it will NOT be saved in the notebook
api_key = getpass.getpass('Enter your API key :')
client = Groq(api_key=api_key)
MODEL = 'meta-llama/llama-4-scout-17b-16e-instruct'

print("Read your key")

Enter your API key :··········
Read your key


In [44]:

import os

CSV_PATH = None
for candidate in [
    '/content/series-200526.csv',
    '/content/drive/MyDrive/series-200526.csv',
]:
    if os.path.exists(candidate):
        CSV_PATH = candidate
        break

if CSV_PATH is None:
    raise FileNotFoundError(
        'Upload series-200526.csv via Files panel or mount Google Drive'
    )


raw = pd.read_csv(CSV_PATH)
raw.columns = ['Period', 'CPIH_Rate']
data_all = raw.iloc[7:].copy()
data_all['CPIH_Rate'] = pd.to_numeric(data_all['CPIH_Rate'], errors='coerce')
data_all = data_all.dropna(subset=['CPIH_Rate']).reset_index(drop=True)

annual    = data_all[data_all['Period'].str.match(r'^\d{4}$')].copy()
quarterly = data_all[data_all['Period'].str.contains('Q')].copy()
monthly   = data_all[data_all['Period'].str.match(r'^\d{4} [A-Z]{3}$')].copy()


monthly['Date'] = pd.to_datetime(monthly['Period'], format='%Y %b')
monthly = monthly.sort_values('Date').reset_index(drop=True)

annual['Year'] = annual['Period'].astype(int)
annual = annual.sort_values('Year').reset_index(drop=True)

print(f'Loaded: {len(annual)} annual | {len(quarterly)} quarterly | {len(monthly)} monthly readings')
print(f'Monthly range: {monthly["Period"].iloc[0]}  →  {monthly["Period"].iloc[-1]}')
print(f'Annual range : {annual["Year"].min()}  →  {annual["Year"].max()}')

Loaded: 37 annual | 149 quarterly | 448 monthly readings
Monthly range: 1989 JAN  →  2026 APR
Annual range : 1989  →  2025


In [45]:

def build_data_context() -> str:
    """Returns a compact JSON-like text summary of all data for the system prompt."""
    annual_records = annual[['Year', 'CPIH_Rate']].rename(
        columns={'Year': 'year', 'CPIH_Rate': 'rate'}
    ).to_dict('records')

    monthly_records = monthly[['Period', 'CPIH_Rate']].rename(
        columns={'Period': 'period', 'CPIH_Rate': 'rate'}
    ).to_dict('records')

    return f"""DATASET: UK CPIH Annual Inflation Rate (%, ONS series L55O, base year 2015=100)
SOURCE FILE: series-200526.csv  |  RELEASE DATE: 20-05-2026
UNIT: % annual rate of change
GRANULARITIES AVAILABLE: annual (1989-2025), quarterly (1989 Q1 - 2025 Q4), monthly (1989 JAN - 2026 APR)

ANNUAL DATA (37 rows):
{annual_records}

MONTHLY DATA (448 rows, last 30 shown for brevity):
{monthly_records[-30:]}

FULL MONTHLY DATA for aggregations:
{monthly_records}
"""

DATA_CONTEXT = build_data_context()
print(f'Data context built: {len(DATA_CONTEXT):,} characters')

Data context built: 19,152 characters


In [46]:
PALETTE = {
    'bg': '#0f1117',
    'panel': '#1a1d27',
    'line': '#7c6aff',
    'bar': '#7c6aff',
    'accent': '#ff6b8a',
    'text': '#e8e8f0',
    'grid': '#2a2d3e',
    'highlight': '#ffd166',
}

def _style_ax(ax, fig):
    fig.patch.set_facecolor(PALETTE['bg'])
    ax.set_facecolor(PALETTE['panel'])
    ax.tick_params(colors=PALETTE['text'], labelsize=9)
    ax.xaxis.label.set_color(PALETTE['text'])
    ax.yaxis.label.set_color(PALETTE['text'])
    ax.title.set_color(PALETTE['text'])
    for spine in ax.spines.values():
        spine.set_edgecolor(PALETTE['grid'])
    ax.grid(axis='y', color=PALETTE['grid'], linewidth=0.6, linestyle='--', alpha=0.7)


def chart_trend_monthly(start_year: int = 1989, end_year: int = 2026,
                         highlight_period: str = None) -> str:
    """Monthly trend line, return base64 PNG."""
    subset = monthly[
        (monthly['Date'].dt.year >= start_year) &
        (monthly['Date'].dt.year <= end_year)
    ]
    fig, ax = plt.subplots(figsize=(10, 4))
    _style_ax(ax, fig)
    ax.plot(subset['Date'], subset['CPIH_Rate'],
            color=PALETTE['line'], linewidth=1.5, zorder=3)
    ax.fill_between(subset['Date'], subset['CPIH_Rate'],
                    alpha=0.15, color=PALETTE['line'])

    peak = subset.loc[subset['CPIH_Rate'].idxmax()]
    ax.scatter(peak['Date'], peak['CPIH_Rate'],
               color=PALETTE['accent'], zorder=5, s=60)
    ax.annotate(f" Peak: {peak['CPIH_Rate']}% ({peak['Period']})",
                xy=(peak['Date'], peak['CPIH_Rate']),
                color=PALETTE['accent'], fontsize=8.5, va='bottom')
    ax.axhline(2, color=PALETTE['highlight'], linewidth=0.8,
               linestyle=':', alpha=0.8, label='2% target')
    ax.set_title(f'UK CPIH Monthly Inflation Rate ({start_year}–{end_year})', fontsize=12)
    ax.set_ylabel('CPIH Rate (%)')
    ax.legend(facecolor=PALETTE['panel'], labelcolor=PALETTE['text'], fontsize=8)
    fig.tight_layout()
    return _fig_to_b64(fig)


def chart_annual_bar(start_year: int = 1989, end_year: int = 2025,
                      highlight_years: list = None) -> str:
    """Annual bar chart, return base64 PNG."""
    subset = annual[(annual['Year'] >= start_year) & (annual['Year'] <= end_year)]
    fig, ax = plt.subplots(figsize=(12, 4))
    _style_ax(ax, fig)
    colors = [PALETTE['accent'] if (highlight_years and y in highlight_years)
              else PALETTE['bar'] for y in subset['Year']]
    ax.bar(subset['Year'], subset['CPIH_Rate'], color=colors,
           edgecolor=PALETTE['bg'], linewidth=0.4, zorder=3)
    ax.axhline(2, color=PALETTE['highlight'], linewidth=0.9,
               linestyle=':', alpha=0.9, label='2% target')
    ax.set_title(f'UK CPIH Annual Inflation Rate ({start_year}–{end_year})', fontsize=12)
    ax.set_ylabel('CPIH Rate (%)')
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.xticks(rotation=45, ha='right', fontsize=8)
    ax.legend(facecolor=PALETTE['panel'], labelcolor=PALETTE['text'], fontsize=8)
    fig.tight_layout()
    return _fig_to_b64(fig)


def chart_comparison_bar(labels: list, values: list, title: str) -> str:
    """Generic labelled bar comparison, return base64 PNG."""
    fig, ax = plt.subplots(figsize=(max(6, len(labels) * 0.9), 4))
    _style_ax(ax, fig)
    bars = ax.bar(range(len(labels)), values,
                  color=[PALETTE['accent'] if v == max(values) else PALETTE['bar']
                         for v in values],
                  edgecolor=PALETTE['bg'], linewidth=0.4, zorder=3)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
    ax.bar_label(bars, fmt='%.1f%%', padding=3,
                 color=PALETTE['text'], fontsize=8)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('CPIH Rate (%)')
    fig.tight_layout()
    return _fig_to_b64(fig)


def _fig_to_b64(fig) -> str:
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close(fig)
    buf.seek(0)
    return 'data:image/png;base64,' + base64.b64encode(buf.read()).decode()



In [47]:
SYSTEM_PROMPT = """
You are a precise data analyst. Your ONLY data source is the UK CPIH dataset below.

{data_context}

==== RULES ====
1. ONLY answer questions that can be computed from the dataset above.
2. If the question requires data NOT in this dataset (e.g. unemployment, GDP, RPI, CPI without housing, stock prices, other countries, forecasts beyond Apr 2026), respond with:
   OUT_OF_SCOPE: <one sentence explanation>
3. If the question IS answerable, produce a JSON block ONLY (no prose before or after), in this exact format:
{{
  "answer": "<natural-language answer with the key number(s)>",
  "numbers": {{"key_stat": value_or_string, ...}},
  "code": "<pandas Python snippet that produces the answer — 3-8 lines>",
  "chart_type": "trend_monthly" | "annual_bar" | "comparison_bar" | "none",
  "chart_params": {{ ... }}  // for trend_monthly: start_year, end_year (ints);
                             // for annual_bar: start_year, end_year, highlight_years (list of ints);
                             // for comparison_bar: labels (list of str), values (list of float), title (str);
                             // for none: {{}}
}}
4. Numbers must come from the dataset. Do NOT invent figures.
5. For trends over multiple years, prefer trend_monthly chart.
6. For single-year monthly breakdown or small comparisons, prefer comparison_bar.
7. For whole-history annual view, prefer annual_bar.
""".format(data_context=DATA_CONTEXT)

print(f'System prompt built: {len(SYSTEM_PROMPT):,} chars')

System prompt built: 20,520 chars


In [48]:
import json, time

def ask_cpih(question: str) -> dict:
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=1500,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ]
    )
    raw = response.choices[0].message.content.strip()


    if 'OUT_OF_SCOPE:' in raw:
        idx = raw.index('OUT_OF_SCOPE:')
        return {
            'out_of_scope': True,
            'answer': raw[idx + len('OUT_OF_SCOPE:'):].strip(),
            'code': '', 'chart_b64': None
        }

    # ── Extract JSON block ──
    # Find outermost { ... }
    brace_start = raw.find('{')
    depth, brace_end = 0, -1
    for i, ch in enumerate(raw):
        if ch == '{': depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                brace_end = i
                break

    if brace_start == -1 or brace_end == -1:
        return {'out_of_scope': False,
                'answer': f'Model did not return valid JSON. Raw: {raw[:300]}',
                'code': '', 'chart_b64': None}

    clean = raw[brace_start:brace_end+1]
    clean = re.sub(r',\s*([}\]])', r'\1', clean)

    try:
        parsed = json.loads(clean)
    except json.JSONDecodeError:
        return {'out_of_scope': False,
                'answer': f'JSON parse failed. Raw: {raw[:300]}',
                'code': '', 'chart_b64': None}

    parsed['out_of_scope'] = False
    ct = parsed.get('chart_type', 'none')
    cp = parsed.get('chart_params', {})
    try:
        if ct == 'trend_monthly':
            parsed['chart_b64'] = chart_trend_monthly(
                start_year=cp.get('start_year', 1989),
                end_year=cp.get('end_year', 2026)
            )
        elif ct == 'annual_bar':
            parsed['chart_b64'] = chart_annual_bar(
                start_year=cp.get('start_year', 1989),
                end_year=cp.get('end_year', 2025),
                highlight_years=cp.get('highlight_years', [])
            )
        elif ct == 'comparison_bar':
            parsed['chart_b64'] = chart_comparison_bar(
                labels=cp.get('labels', []),
                values=cp.get('values', []),
                title=cp.get('title', '')
            )
        else:
            parsed['chart_b64'] = None
    except Exception as e:
        parsed['chart_b64'] = None

    return parsed

print('ask_cpih() ready')

ask_cpih() ready


## Note : Do not spam the questions , wait for a few seconds before asking another one or re-run the cell below for better chart outputs sometimes it may also return "Answer: Model did not return valid JSON. Raw {Answer}", its on free tier has rate limits. Please wait 5–10 seconds between questions to avoid hitting the limit. If you see a 429 error, simply wait a moment and click Ask again button.


In [49]:
EXAMPLE_QUESTIONS = [
    'What was the CPIH inflation rate in 2022?',
    'Which month had the highest inflation on record?',
    'Show me the inflation trend from 2019 to 2026.',
    'What was the average annual CPIH over 2020, 2021, and 2022?',
    'What was UK unemployment in 2022?',
]

CSS = """
.answer-box { background:#1a1d27; border:1px solid #2a2d3e; border-radius:8px;
              padding:16px; color:#e8e8f0; font-size:15px; }
.code-box   { background:#0f1117; border:1px solid #2a2d3e; border-radius:8px;
              padding:12px; color:#a8dadc; font-family:monospace; font-size:12px; }
.oos-box    { background:#2d1a2e; border:1px solid #9b2335; border-radius:8px;
              padding:16px; color:#ff6b8a; font-size:15px; }
.title-row  { text-align:center; }
"""

def handle_question(question: str):
    if not question.strip():
        return '<div class="answer-box">Please enter a question.</div>', '', None
    try:
        result = ask_cpih(question)
    except Exception as e:
        return (f'<div class="oos-box"> API error: {str(e)[:200]}</div>',
                '', None)

    if result.get('out_of_scope'):
        answer_html = (f'<div class="oos-box">'
                       f'<strong>Out of scope</strong><br><br>'
                       f'{result["answer"]}'
                       f'</div>')
        return answer_html, '# Not answerable from CPIH dataset.', None

    answer = result.get('answer', '')
    numbers = result.get('numbers', {})
    nums_html = ''.join(
        f'<span style="background:#2a2d3e;padding:4px 10px;'
        f'border-radius:5px;margin:3px;display:inline-block;'
        f'color:#7c6aff;"><b>{k}</b>: {v}</span>'
        for k, v in numbers.items()
    )
    answer_html = (f'<div class="answer-box">'
                   f'<b>Answer:</b> {answer}'
                   f'{"<br><br>" + nums_html if nums_html else ""}'
                   f'</div>')
    code_text = result.get('code', '')

    chart_img = None
    chart_b64 = result.get('chart_b64')
    if chart_b64:
        try:
            from PIL import Image
            header, data = chart_b64.split(',', 1)
            chart_img = Image.open(io.BytesIO(base64.b64decode(data)))
        except Exception:
            chart_img = None

    return answer_html, code_text, chart_img


with gr.Blocks(css=CSS, title='UK CPIH Analyser') as demo:
    gr.HTML('<div class="title-row"><h1>🇬🇧 UK CPIH Inflation Analyser</h1>'
            '<p style="color:#888">Powered by Claude · Data: ONS series-200526.csv (released 20 May 2026)</p></div>')

    with gr.Row():
        question_box = gr.Textbox(
            label='Ask a question about UK CPIH inflation',
            placeholder='e.g. What was inflation in 2022?',
            lines=2, scale=4
        )
        submit_btn = gr.Button('Ask →', variant='primary', scale=1)

    gr.Examples(
        examples=[[q] for q in EXAMPLE_QUESTIONS],
        inputs=question_box,
        label='Example questions (click to load)'
    )

    answer_out = gr.HTML(label='Answer')
    chart_out  = gr.Image(label='Chart', type='pil')
    code_out   = gr.Code(label='Provenance — pandas code that computed this answer',
                          language='python')

    submit_btn.click(
        fn=handle_question,
        inputs=question_box,
        outputs=[answer_out, code_out, chart_out]
    )
    question_box.submit(
        fn=handle_question,
        inputs=question_box,
        outputs=[answer_out, code_out, chart_out]
    )

    gr.HTML('<hr><p style="color:#555;text-align:center;font-size:12px">'
            'This tool only answers questions derivable from the ONS CPIH dataset. '
            'Out-of-scope queries (GDP, unemployment, RPI, other countries) are refused.</p>')

demo.launch(share=True, debug=False)

/tmp/ipykernel_6447/1213278258.py:62: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, title='UK CPIH Analyser') as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://857f180e8b4f15fdf2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
